# FAO GAEZ Land Use Maps Generator
Some outputs from the original data are note ok, so these will be replaced using FAO GAEZ data, including:

|Crop|Reason|
|:---|:-----|
|Oil Palm | Land use raster wrong, missing important production zones like Indonesia|
|Tomatoes| SPAM yields overly high, distorting results for some countries |
|Potatoes| Same as tomatoes |
| Maize | Super high yields |
|Tobacco| High yields |

## Suitability Index Inventory
We start by generating a file with all the files we have

In [57]:
import os
import polars as pl
from sbtn_leaf.paths import data_path
from pathlib import Path

project_root = Path.cwd().parents[1]   # adjust number

In [58]:
files_folder = "../data/crops/FAO_GAEZ/SuitIndex"
files_prefix = "GAEZ-V5.RES05-SIX.HP0120.AGERA5.HIST."
files_extension = ".tif"
TEC_LOOKUP = {
    "HILM": "high_input_irrigated",
    "HRLM": "high_input_rainfed",
    "LILM": "low_input_irrigated",
    "LRLM": "low_input_rainfed",
}

IRR_LOOKUP = {
    "HILM": "irr",
    "HRLM": "rf",
    "LILM": "irr",
    "LRLM": "rf",
}

In [59]:
crop_index = pl.read_csv(data_path("crops", "FAO_GAEZ", "FAO_GAEZ_croplist.csv")).drop(["description", "index"]).rename({"caption":"crop_name", "code": "crop_code"})

In [60]:
crop_index.head()

crop_code,crop_name
str,str
"""ALF""","""Alfalfa"""
"""BAN""","""Banana"""
"""BRL""","""Barley"""
"""BSG""","""Biomass sorghum"""
"""BCH""","""Brachiaria"""


In [61]:
folder_path = data_path("crops", "FAO_GAEZ", "SuitIndex")

In [62]:
output_rows = []

for file in os.listdir(folder_path):
    if not file.lower().endswith(".tif"):  # skip not tif files
        continue

    filename = Path(file).stem
    file_path = str(Path(file).resolve())
    parts = filename.split(".")
    if len(parts) < 2:
            continue
    
    crop_code = parts[-2].strip().upper()
    tech_code = parts[-1].strip().upper()

    output_rows.append({
        "filename": file,
        "filepath": file_path,
        "crop_code": crop_code,
        "tech_code": tech_code,
        "technology": TEC_LOOKUP.get(tech_code),
        "irrigation_practice": IRR_LOOKUP.get(tech_code)
    })

df = pl.DataFrame(output_rows)

# Attach crop names
df = df.join(
     other=crop_index,
     on="crop_code",
     how="left"
)

In [63]:
df.head(2)

filename,filepath,crop_code,tech_code,technology,irrigation_practice,crop_name
str,str,str,str,str,str,str
"""GAEZ-V5.RES05-SIX.HP0120.AGERA…","""C:\Users\loyola\OneDrive - Wor…","""ALF""","""HILM""","""high_input_irrigated""","""irr""","""Alfalfa"""
"""GAEZ-V5.RES05-SIX.HP0120.AGERA…","""C:\Users\loyola\OneDrive - Wor…","""ALF""","""HRLM""","""high_input_rainfed""","""rf""","""Alfalfa"""


Storing it

In [64]:
csv_out_path = project_root / "data" / "crops" / "FAO_GAEZ" / "fao_gaez_crop_suitindex_list.csv"

df.write_csv(
    str(csv_out_path),
    separator=";"
)

In [68]:
fao_gaez_crops = df["crop_name"].unique()

## Attaching to our analyzed crops

In [65]:
sbtn_crops = pl.read_excel(data_path("crops","FAO_GAEZ", "crops_file_index.xlsx"))

There a few crop names that need adjustment from FAO GAEZ

In [72]:
CROP_RENAME = {
    "Citrus": "Orange",
    "Sugar cane": "Sugarcane",
    "White potato": "Potato",
}

crop_renaming = pl.DataFrame(
    {"fao_gaez_name": ["Citrus","Sugar cane", "White potato"],
    "sbtn_name": ["Orange", "Sugarcane", "Potato"]}
)

Some don't have matches and are kept the same, including:
- Apples
- Grapes

Adding the suitability index file

In [66]:
sbtn_crops = sbtn_crops.join(
    other=df,
    how="left",
    on = ["crop_name", "irrigation_practice"]
)

In [67]:
sbtn_crops.head()

crop_name,crop_practice_string,crop_type,irrigation_practice,filename,filepath,crop_code,tech_code,technology
str,str,str,str,str,str,str,str,str
"""Barley""","""irr_ron""","""annual""","""irr""","""GAEZ-V5.RES05-SIX.HP0120.AGERA…","""C:\Users\loyola\OneDrive - Wor…","""BRL""","""HILM""","""high_input_irrigated"""
"""Barley""","""irr_ron""","""annual""","""irr""","""GAEZ-V5.RES05-SIX.HP0120.AGERA…","""C:\Users\loyola\OneDrive - Wor…","""BRL""","""LILM""","""low_input_irrigated"""
"""Barley""","""irr_roff""","""annual""","""irr""","""GAEZ-V5.RES05-SIX.HP0120.AGERA…","""C:\Users\loyola\OneDrive - Wor…","""BRL""","""HILM""","""high_input_irrigated"""
"""Barley""","""irr_roff""","""annual""","""irr""","""GAEZ-V5.RES05-SIX.HP0120.AGERA…","""C:\Users\loyola\OneDrive - Wor…","""BRL""","""LILM""","""low_input_irrigated"""
"""Cabbage""","""irr""","""annual""","""irr""","""GAEZ-V5.RES05-SIX.HP0120.AGERA…","""C:\Users\loyola\OneDrive - Wor…","""CAB""","""HILM""","""high_input_irrigated"""
